# 股票回测框架 Quickstart

本 Notebook 演示如何使用 `stock-backtest` 进行完整的数据获取、持久化、回测和可视化。

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data import DataManager, DataStorage
from backtest.engine import BacktestEngine
from strategies.base import BuyAndHoldStrategy, MovingAverageCrossStrategy
from visualization import plot_backtest_result, plot_price_with_signals, plot_strategy_comparison

## 1. 获取股票数据

In [ ]:
manager = DataManager()

# 获取平安银行（000001.SZ）2024 年日线数据
df = manager.get_daily_data(
    symbol='000001.SZ',
    start='2024-01-01',
    end='2024-12-31',
    source='auto'
)
print(f"获取到 {len(df)} 条记录")
df.head()

## 2. 数据持久化（SQLite / Parquet）

In [ ]:
# 使用 SQLite 存储
storage = DataStorage(backend='sqlite', db_path='../data/stock_data.db')

# 保存数据
storage.save_daily_data('000001.SZ', df)

# 从本地加载
df_loaded = storage.load_daily_data('000001.SZ', start='2024-06-01', end='2024-12-31')
print(f"从 SQLite 加载 {len(df_loaded)} 条记录")
df_loaded.tail()

In [ ]:
# 使用 Parquet 存储（适合大量数据）
pq_storage = DataStorage(backend='parquet', data_dir='../data/parquet')
pq_storage.save_daily_data('000001.SZ', df)

pq_storage.list_symbols()

## 3. 运行买入持有策略

In [ ]:
engine = BacktestEngine(initial_cash=100000, commission=0.0003)
strategy = BuyAndHoldStrategy()

result = engine.run(strategy, df)
print(f"总收益率: {result['total_return_pct']}")
print(f"最大回撤: {result['max_drawdown_pct']}")
print(f"夏普比率: {result['sharpe_ratio']:.2f}")
print(f"交易次数: {result['total_trades']}")

## 4. 运行均线交叉策略

In [ ]:
engine2 = BacktestEngine(initial_cash=100000, commission=0.0003)
strategy2 = MovingAverageCrossStrategy(short_window=10, long_window=30)

result2 = engine2.run(strategy2, df)
print(f"总收益率: {result2['total_return_pct']}")
print(f"最大回撤: {result2['max_drawdown_pct']}")
print(f"夏普比率: {result2['sharpe_ratio']:.2f}")
print(f"交易次数: {result2['total_trades']}")

## 5. 可视化回测结果

In [ ]:
# 绘制买入持有策略结果
plot_backtest_result(result, show_trades=True, backend='matplotlib')

In [ ]:
# 绘制均线交叉策略的价格与交易信号
plot_price_with_signals(df, result2['trades'])

In [ ]:
# 对比两个策略
plot_strategy_comparison([result, result2], ['买入持有', '均线交叉'])

## 6. 保存回测结果

In [ ]:
# 保存到 SQLite
storage.save_backtest_result('buy_and_hold_000001', result)
storage.save_backtest_result('ma_cross_000001', result2)

# 加载历史结果
loaded_result = storage.load_backtest_result('ma_cross_000001')
print(f"加载结果: {loaded_result['name']}, 总收益: {loaded_result['total_return']*100:.2f}%")

## 7. 获取实时行情（Sina）

In [ ]:
quote = manager.sina.get_realtime_quote('000001.SZ')
quote